<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week2_supervised/Week2_Lesson3_Lecture.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 IB CS — Week 2 · Lesson 3 (Lecture)
## Classic algorithms, metrics, and the rest of A4.2.2/A4.2.3
### A4.2.2, A4.2.3, A4.3.1, A4.3.2, A4.3.3 (HL Focus)

> ⏱ Duration: 2 academic hours.
> 📘 Sources: Baumgarten (Hodder IBDP) + MacKenty & Stephenson (Oxford 2025).

---

### 🎯 What we cover in this lesson (syllabus):

| Syllabus statement | Topic | Command term |
|---|---|---|
| **A4.2.2** | Feature selection | *Describe* |
| **A4.2.3** | Dimensionality reduction | *Describe* |
| **A4.3.1** | Linear regression | *Explain* |
| **A4.3.2** | KNN + Decision Trees | *Explain* |
| **A4.3.3** | Hyperparameter tuning | *Explain* |
| **+** | Metrics (Accuracy/Precision/Recall/F1) | part of A4.3.3 |
| **+** | Overfitting / Underfitting | part of A4.3.3 |

---

### ⚠️ Before we start

> Last week we covered **A4.2.1** (data cleaning). Today we finish **A4.2 (Data Preprocessing — HL)** by covering **A4.2.2** and **A4.2.3** first. Then we move into **A4.3 (ML Approaches — HL)**.
>
> 💎 This is roughly **half of the IB ML content**. Without clear understanding here, a 6+ is unlikely.


## 📌 Part 1 · A4.2.2 Feature Selection (HL)

> **Definition (learn this):** *Feature selection refers to taking care to select only the most relevant features for use in your machine learning models.*
>
> **Feature:** *a numeric property (variable) that can be used as input for a machine learning algorithm.*

### Why do we need it?

Imagine you have 500 features for 100 records. That is:
- 🐌 **Slow** — the model trains for hours instead of minutes
- 🎯 **Less accurate** — the model learns noise and overfits
- 🤯 **Hard to interpret** — it is difficult to see what matters

> 🚨 **Common mistake (Baumgarten, direct quote):**
> *"Don't underestimate the importance of feature selection and engineering. Good features are often more important than the choice of model itself."*

### 🔧 Three feature selection methods (syllabus):

| Method | How it works | When to use it |
|---|---|---|
| **Filter methods** | a statistical metric, such as **Pearson's r**, ranks features independently of the model | large datasets, quick first filtering |
| **Wrapper methods** | try feature subsets, train a model on each, choose the best subset | small datasets, accuracy matters |
| **Embedded methods** | feature selection happens inside training, such as regularisation in Lasso | compromise between filter and wrapper |

> 💎 **Secret #1:** for *"Distinguish between filter and wrapper methods"*, the main difference is **computational cost**:
> - **Filter** — cheap, no model training
> - **Wrapper** — expensive, model is trained many times
> - **Embedded** — balanced, model training and feature selection happen together


In [ ]:
# Demonstration: filter method on a simple dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# Create a dataset: predict house price
n = 200
df = pd.DataFrame({
    'area_m2':      np.random.normal(120, 40, n),
    'bedrooms':     np.random.choice([1,2,3,4,5], n),
    'random_noise': np.random.normal(0, 1, n),       # useless feature
    'age_years':    np.random.exponential(15, n),
    'temperature':  np.random.normal(20, 5, n),      # another useless feature, no link to price
})
df['price'] = (df['area_m2'] * 3.5 + df['bedrooms']*25 - df['age_years']*1.2
               + np.random.normal(0, 20, n))

# Pearson's r: filter method
correlations = df.corr()['price'].drop('price').abs().sort_values(ascending=False)
print("=== Pearson |r| with target variable 'price' ===")
print(correlations.round(3))
print("\n💡 Filter method says: keep area_m2, bedrooms, age_years")
print("    Drop random_noise and temperature because their |r| is close to 0")


In [ ]:
# Visualisation: correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Correlation matrix — basis of the filter method', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


> ⚠️ **Trap (Baumgarten, p. 28):**
> *"As the Spurious Correlations website demonstrates, if you compare enough unrelated data sets, you will find correlations that are, in fact, not real."*
>
> In the IB exam, this is a useful extra point: mention that correlation is not causation. Filter methods can mislead.

### 🎯 Mini-question for class
> *"Outline ONE limitation of filter methods compared to wrapper methods."* **[2]**
>
> 💡 Hint: filter methods do not consider **interactions between features**. If features A and B are weak separately but strong together, a filter method may drop both.


## 📌 Part 2 · A4.2.3 Dimensionality Reduction (HL)

> **Definition (learn this):** *Dimensionality reduction is a group of techniques that reduce the number of variables/features in a data set while preserving as much relevant information as possible.*

### ⚠️ Curse of Dimensionality

> 📘 **Direct quote from MacKenty/Stephenson:**
> *"In high-dimensional space, the feature space grows exponentially. With so many dimensions, each data point becomes an isolated point, far from others. Essentially, every observation appears unique, with few obvious 'near neighbours'."*

**What happens as feature count grows:**

| Dimensionality | Effect |
|---|---|
| 2-3 | Fine, can be visualised |
| 10-50 | Points start to spread out |
| 100+ | All points seem similarly far apart; the model struggles to find patterns |
| 1000+ | **Exponentially** more data is needed |

> 💎 **Secret #2:** for *"Explain the curse of dimensionality"*, mention:
> 1. Exponential growth of feature space
> 2. Sparsity of data points
> 3. Distance-based algorithms such as **KNN** stop working well
> 4. **Exponentially** more data is needed


In [ ]:
# Visualisation: curse of dimensionality
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
np.random.seed(0)

for ax, dim in zip(axes, [2, 10, 100]):
    # Generate 50 points in N-dimensional space
    points = np.random.uniform(0, 1, (50, dim))
    # Calculate all pairwise distances
    from scipy.spatial.distance import pdist
    distances = pdist(points)
    ax.hist(distances, bins=20, color='steelblue', edgecolor='black')
    ax.set_title(f'Dimensions = {dim}\nmin={distances.min():.2f}, max={distances.max():.2f}\n'
                 f'ratio max/min = {distances.max()/distances.min():.1f}',
                 fontsize=11)
    ax.set_xlabel('Distance between points')
    ax.set_ylabel('Frequency')

plt.suptitle('Curse of Dimensionality: as dimensions grow, distances become similar',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


> 🔬 **Observation:** in 100-dimensional space, **all points are almost equally distant** from each other. KNN stops working well because "nearest neighbour" becomes much less meaningful.

### 🛠 Two dimensionality reduction techniques

| Technique | What it does | What it preserves |
|---|---|---|
| **Feature selection** | removes features | original meaning of features |
| **Feature extraction** (for example, **PCA**) | creates new features as combinations of old ones | maximum variance |

> ⚠️ **Important for IB:** *"Statistical techniques such as PCA and LDA are beyond the scope of this course."* (MacKenty/Stephenson p. 262)
>
> 💎 So you should **know the names PCA and LDA** because they may appear in *Identify* questions, but you do **not** need to derive formulas. Understand that they **reduce** dimensionality.

### 📊 Benefits of dimensionality reduction (for IB answers):

1. **Better visualisation** — data can be plotted in 2D/3D
2. **Less overfitting** — the model learns less noise
3. **Less time and memory** — training is faster
4. **Simpler analysis** — patterns are easier to see

> 💎 **Secret #3 (Baumgarten common mistake):**
> *"Dimensionality reduction does not ALWAYS lead to better model performance. The primary goal is to simplify the model, which can help but might also lead to LOSS of critical information."*
>
> In *Discuss* questions, always mention the trade-off: simplicity vs information loss.


## 📈 Part 3 · A4.3.1 Linear Regression

> **Definition:** *Linear regression is a machine learning algorithm that seeks a linear line of best fit for a given data set, from which extrapolations can be made.*

### Formula (know this):

$$Y = \beta_0 + \beta_1 X + \epsilon$$

| Symbol | Meaning |
|---|---|
| $Y$ | dependent variable: what we predict |
| $X$ | independent variable: what we use for prediction |
| $\beta_0$ | **intercept** — value of Y when X=0 |
| $\beta_1$ | **slope** — how much Y changes when X increases by 1 |
| $\epsilon$ | error term |

> 💎 **Secret #4:** examiners often ask *"Describe the significance of slope and intercept"* **[4]**.
> Full answer:
> - **Intercept** — starting point / baseline value
> - **Slope** — how much Y changes when X changes; positive slope means a positive relationship
> - **Magnitude of slope** — strength of the effect
> - **Concrete example** — e.g. "intercept = baseline house price, slope = extra cost per square metre"

### When to use it?

| Suitable | Not suitable |
|---|---|
| Linear relationship between X and Y | Non-linear patterns; use polynomial regression or another model |
| Numeric continuous output | Categorical output; use classification |
| Interpretability matters | Complex patterns; neural networks may fit better |


In [ ]:
# Demo: linear regression on house prices
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Data: house area → price
np.random.seed(0)
area = np.random.uniform(50, 300, 50)
price = 178220 + 1603 * area + np.random.normal(0, 30000, 50)

X = area.reshape(-1, 1)
y = price

model = LinearRegression()
model.fit(X, y)

slope = model.coef_[0]
intercept = model.intercept_
r2 = r2_score(y, model.predict(X))

print(f"Intercept (β₀) = {intercept:,.0f}  → starting price when area is 0")
print(f"Slope (β₁)     = {slope:,.0f}  → extra price per square metre")
print(f"R² (R-squared) = {r2:.3f}  → proportion of explained variance")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(area, price, alpha=0.7, color='steelblue', label='Real data')
x_line = np.linspace(50, 300, 100)
y_line = intercept + slope * x_line
ax.plot(x_line, y_line, 'r-', linewidth=2, label=f'Y = {intercept:,.0f} + {slope:,.0f}·X')
ax.set_xlabel('Area (square metres)')
ax.set_ylabel('Price ($)')
ax.set_title(f'Linear Regression: predicting house price (R²={r2:.3f})',
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 📊 R² (R-squared) — the main metric for regression

> **Definition:** *R-squared is a statistical measure that indicates how well the linear regression model fits the data points.*

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$$

| R² value | Interpretation |
|---|---|
| **0** | the model explains **none** of the variation |
| **0.5** | explains 50% of variation, moderately useful |
| **0.8+** | good fit |
| **1.0** | **perfect** explanation, often suspicious and possibly overfitting |

> 💎 **Secret #5:** R² very close to 1.0 is **suspicious** and may mean overfitting. For *"How would you assess accuracy of a regression model?"*, mention R² **and also** MSE, MAE, or mean absolute percentage error.


## 🎯 Part 4 · A4.3.2 Classification — KNN and Decision Trees

> **Definition:** *Classification techniques are where a machine learning model has been trained to identify, from a predefined list of categories, which category the input data would most likely belong to.*

**Two main non-neural classifiers:**

### 1️⃣ K-Nearest Neighbours (KNN)

> **Definition:** *Data points are categorized based on the categories of the nearest points around them; k is a variable representing how many nearest points should "vote".*

**How it works (3 steps from MacKenty/Stephenson):**
1. For a new point, calculate the **distance** to all known points
2. Choose the **K nearest** neighbours
3. Use **majority vote** to assign the class

**Properties of KNN:**
- ✅ **Non-parametric** — does not assume a fixed data shape
- ✅ **Instance-based / Lazy learning** — does not build a model; stores data
- ❌ **Sensitive to scale** — you must **normalise** before using it
- ❌ **Slow at prediction** — calculates distances to all points

> ⚠️ **Top tip from Baumgarten:** *"Choose an ODD number for k when the number of classes is even to avoid tie situations."*


In [ ]:
# Visualisation: how K affects the decision boundary
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_blobs

X_demo, y_demo = make_blobs(n_samples=150, centers=2, cluster_std=1.5, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, k in zip(axes, [1, 5, 25]):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_demo, y_demo)

    # Grid for visualisation
    xx, yy = np.meshgrid(np.linspace(X_demo[:,0].min()-1, X_demo[:,0].max()+1, 200),
                         np.linspace(X_demo[:,1].min()-1, X_demo[:,1].max()+1, 200))
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X_demo[:,0], X_demo[:,1], c=y_demo, cmap='coolwarm',
               edgecolor='k', s=50)
    ax.set_title(f'KNN, k={k}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

plt.suptitle('Effect of K: small K → overfitting · large K → underfitting',
             fontsize=13, y=1.03)
plt.tight_layout(); plt.show()


> 🔬 **Observation:**
> - **k=1**: the boundary is very wiggly, so the model **overfits**
> - **k=5**: a reasonable fit for this example
> - **k=25**: the boundary is too smooth, so the model **underfits**
>
> 💎 **Secret #6:** *"Outline how choice of k affects classification accuracy"* is a standard 2-mark question. Include **both** extremes: small k → overfit; large k → underfit.

---

### 2️⃣ Decision Trees

> **Definition:** *A graphical representation of conditions that result in a classification decision; think of it as a decision-making flowchart that the ML model creates automatically.*

**How it works (MacKenty/Stephenson):**
1. **Root node** — starting node
2. **Split** — data is split by the feature that gives the most homogeneous subsets, using **Gini impurity** or **entropy**
3. **Recursive splitting** — repeat until a stopping condition
4. **Pruning** (optional) — remove unnecessary branches to reduce overfitting

**Properties:**
- ✅ **Eager learner** — builds a model during training
- ✅ Does not require normalisation
- ✅ Highly interpretable, read as if-else rules
- ❌ Prone to **overfitting** if too deep
- ❌ Sensitive to small changes in data


In [ ]:
# Decision tree on the same dataset
from sklearn.tree import DecisionTreeClassifier, plot_tree

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, depth in zip(axes, [1, 3, 10]):
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_demo, y_demo)

    xx, yy = np.meshgrid(np.linspace(X_demo[:,0].min()-1, X_demo[:,0].max()+1, 200),
                         np.linspace(X_demo[:,1].min()-1, X_demo[:,1].max()+1, 200))
    Z = dt.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X_demo[:,0], X_demo[:,1], c=y_demo, cmap='coolwarm',
               edgecolor='k', s=50)
    ax.set_title(f'Decision Tree, max_depth={depth}', fontsize=12, fontweight='bold')

plt.suptitle('Effect of max_depth: depth 1 → underfit · depth 10 → overfit',
             fontsize=13, y=1.03)
plt.tight_layout(); plt.show()


### 🆚 KNN vs Decision Trees — IB comparison table

| Criterion | KNN | Decision Tree |
|---|---|---|
| Learning type | Lazy / Instance-based | Eager |
| Needs normalisation? | **Yes** because it uses distance | No |
| Interpretability | medium | **high**, because it is if-else style |
| Training speed | instant | medium |
| Prediction speed | **slow** | fast |
| Main hyperparameter | **k**, number of neighbours | **max_depth**, depth of tree |
| Main risk | curse of dimensionality | overfitting |
| Best for | small/medium datasets | mixed data types |

> 💎 **Secret #7 (Top tip from Baumgarten):**
> *"When choosing between KNN, decision trees and neural networks:*
> - *KNN suits moderate data sizes and low dimensions*
> - *Decision trees handle mixed data sizes well*
> - *Neural networks excel with large, complex data sets"*
>
> This is a ready-made structure for *Discuss [6]* questions.


## 📐 Part 5 · Evaluation metrics (part of A4.3.3)

### Confusion Matrix — the foundation of classification metrics

> **Definition:** *A simple pictorial means of representing how well a machine learning model is performing.*

|  | Predicted **POSITIVE** | Predicted **NEGATIVE** |
|---|---|---|
| **Actually POSITIVE** | **TP** (true positive) ✅ | **FN** (false negative) ❌ |
| **Actually NEGATIVE** | **FP** (false positive) ❌ | **TN** (true negative) ✅ |

### 4 main metrics (learn the formulas)

| Metric | Formula | When it matters |
|---|---|---|
| **Accuracy** | $\frac{TP+TN}{TP+TN+FP+FN}$ | classes are balanced |
| **Precision** | $\frac{TP}{TP+FP}$ | FP is costly, such as marking an important email as spam |
| **Recall** | $\frac{TP}{TP+FN}$ | FN is costly, such as missing cancer |
| **F1 Score** | $2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$ | balance needed / classes are imbalanced |


In [ ]:
# Worked example from MacKenty/Stephenson (pp. 275-277)
# The model classifies email: spam / not spam
# Out of 100 emails, 60 are actually spam.
# The model predicted 50 as spam, and 40 of those were correct.

TP = 40    # correctly predicted spam
FN = 60 - 40  # missed spam = 20
FP = 50 - 40  # false alarm = 10
TN = 100 - TP - FN - FP  # = 30

print(f"TP (true positive)  = {TP}")
print(f"FN (false negative) = {FN}")
print(f"FP (false positive) = {FP}")
print(f"TN (true negative)  = {TN}")

accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)

print(f"\nAccuracy  = ({TP}+{TN})/100 = {accuracy:.2f} ({accuracy*100:.0f}%)")
print(f"Precision = {TP}/{TP+FP}    = {precision:.2f} ({precision*100:.0f}%)")
print(f"Recall    = {TP}/{TP+FN}    = {recall:.2f} ({recall*100:.0f}%)")
print(f"F1 Score  = 2·{precision:.2f}·{recall:.2f}/({precision:.2f}+{recall:.2f}) = {f1:.2f}")


In [ ]:
# Visualisation: confusion matrix
cm = np.array([[TP, FN], [FP, TN]])

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: Spam', 'Predicted: Not Spam'],
            yticklabels=['Actual: Spam', 'Actual: Not Spam'],
            cbar=False, ax=ax, annot_kws={'size': 16})
ax.set_title('Confusion Matrix — spam classifier', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


### 🎯 When is each metric more important? (KEY IB concept)

| Scenario | Cost of FP | Cost of FN | Main metric |
|---|---|---|---|
| **Spam filter** | important email → spam folder | spam reaches inbox | **Precision** |
| **Cancer diagnosis** | healthy person told "cancer" | sick person told "healthy" | **Recall** |
| **Face identification for arrest** | innocent person arrested | suspect not flagged | **Precision** |
| **Fraudulent transaction detection** | honest customer blocked | fraud missed | **Balance → F1** |

> 💎 **Secret #8 (VERY IMPORTANT for the exam):**
> *"Explain why high Recall is more important than high Accuracy in medicine"* is a typical 4-mark question.
>
> **4/4 answer structure:**
> 1. In medicine, a **false negative** means a missed disease, risking death or worsening health
> 2. **Accuracy** can be high if the disease is rare; a model can always say "healthy" and still have high accuracy, but recall = 0%
> 3. **Recall** directly shows what proportion of sick patients the model finds
> 4. Therefore **low recall = missed diagnoses**, which is unacceptable

> 🚨 **Common mistake (Baumgarten):**
> *"It is common for students to over-rely on accuracy and neglect the nuance provided by the other metrics."*
> Never write only about accuracy in an exam answer.


## ⚖️ Part 6 · Overfitting and Underfitting

> **Overfitting:** *The model memorizes detail from the training data that is too fine-grained to generalize. Performs WELL on training, POORLY on test.*
>
> **Underfitting:** *The model is too simple, hasn't learned enough. Performs POORLY on BOTH training and test.*

### 🎯 How to recognise them (Baumgarten)

| Sign | Overfitting | Underfitting |
|---|---|---|
| Performance on training data | **excellent** | **poor** |
| Performance on test data | **poor** | **poor** |
| Model complexity | too complex: many features, deep tree, k=1 | too simple: k=100, depth=1 |
| Solution | simplify the model | make the model more expressive |

> 💎 **Secret #9:** for *"Describe ONE strategy to prevent overfitting in decision trees"* **[2]**, several answers are valid:
> 1. **Pruning** — cut branches
> 2. **Limit max_depth** — restrict tree depth
> 3. **Min samples per leaf** — require a minimum number of records in a leaf
> 4. **Cross-validation** — evaluate on different subsets


In [ ]:
# Demonstration: training vs test accuracy for different K values
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Generate a slightly more complex dataset
X2, y2 = make_blobs(n_samples=300, centers=2, cluster_std=2.5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X2, y2, test_size=0.3, random_state=42)

ks = range(1, 50, 2)
train_acc = []
test_acc = []

for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, knn.predict(X_train)))
    test_acc.append(accuracy_score(y_test, knn.predict(X_test)))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(ks, train_acc, 'o-', label='Training accuracy', color='steelblue')
ax.plot(ks, test_acc, 's-', label='Test accuracy', color='orange')
ax.axvspan(1, 5, alpha=0.2, color='red', label='Overfitting zone')
ax.axvspan(35, 50, alpha=0.2, color='blue', label='Underfitting zone')
ax.set_xlabel('K (number of neighbours)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('KNN: train/test accuracy gap indicates overfitting',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower left'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"k=1:  train={train_acc[0]:.3f}, test={test_acc[0]:.3f}  → gap = {train_acc[0]-test_acc[0]:.3f} (overfitting)")
print(f"k=49: train={train_acc[-1]:.3f}, test={test_acc[-1]:.3f}  → both low (underfitting)")


## 🎚 Part 7 · A4.3.3 Hyperparameter Tuning

> **Definition:** *Hyperparameters are values set BEFORE the learning process that guide the algorithm; hyperparameter tuning is the experimentation to find the optimal combination.*

**Examples of hyperparameters:**

| Algorithm | Hyperparameter |
|---|---|
| KNN | **k**, number of neighbours |
| Decision Tree | **max_depth**, min_samples_split, min_samples_leaf |
| Neural Network | **learning rate**, activation function, number of hidden layers |
| K-means | **k**, number of clusters |

> 💎 **Secret #10:** do not confuse **parameter** and **hyperparameter**:
> - **Parameter** — *learned from data*, such as neural-network weights
> - **Hyperparameter** — *set before training*, such as learning rate

### Tuning methods:
1. **Manual search** — try values yourself
2. **Grid Search** — systematic search over all combinations
3. **Random Search** — random combinations, often faster than grid
4. **Cross-validation** — evaluation across K different subsets; gold standard for the IB IA


## 📝 Part 8 · Mini Exam Practice (in class)

### Question 1 (Baumgarten Review #7, p. 38)
> *A maintenance system uses supervised learning to forecast equipment failures in an industrial plant.*
>
> **a)** *Define* "precision" and "recall" in this context. **[2]**
> **b)** *Explain* why F1 score is a better measure than accuracy when **false negatives have higher costs**. **[3]**
> **c)** *Describe* how a confusion matrix can be used to visually illustrate model success. **[3]**

> 💎 **Breakdown for (b) [3]:**
> 1. F1 score is the **harmonic mean** of precision and recall
> 2. Accuracy can be misleading in **imbalanced datasets**
> 3. When FN are costly, such as missing equipment failure, high **recall** is critical, and F1 includes recall directly

---

### Question 2 (Baumgarten Review #4, p. 38)
> *A medical research institution develops a decision tree model to classify patients into risk categories for heart disease.*
>
> **a)** *Describe one advantage* of using decision trees for this classification. **[2]**
> **b)** *Describe one disadvantage*. **[2]**
> **c) i)** *Identify* one critical parameter that could impact performance. **[1]**
> **c) ii)** *Outline* its role. **[2]**

> 💎 **Hints:**
> - (a): high interpretability; a doctor can see **why** the model made a decision, which matters in medicine
> - (b): prone to overfitting; sensitive to small changes in data
> - (c.i): **max_depth** or min_samples_split
> - (c.ii): max_depth limits tree depth; too large → overfit; too small → underfit

---

### Question 3 (MacKenty/Stephenson End-of-topic #10, p. 315)
> **a)** *Describe* the role of feature selection. **[2]**
> **b)** *Outline* the three different feature selection strategies. **[6]**

> 💎 **Structure for (b) [6]:** 2 marks per method (filter, wrapper, embedded): what it does + one advantage or limitation.


## ✅ Checklist after Lesson 3

By the next lesson, the workshop, you should be able to:

- [ ] Know **3 feature selection methods** (filter / wrapper / embedded) and their trade-offs
- [ ] Explain the **curse of dimensionality** in 4 points
- [ ] Know the linear regression formula `Y = β₀ + β₁X + ε` and the meaning of each term
- [ ] Interpret **R²**
- [ ] Understand the **KNN** algorithm (3 steps) and why normalisation matters
- [ ] Understand **Decision Trees** and ways to reduce overfitting
- [ ] Know the **4 metric formulas** (Accuracy, Precision, Recall, F1) **by heart**
- [ ] Draw a **Confusion Matrix** and label TP/TN/FP/FN
- [ ] Explain **when each metric matters most** (medicine → recall, etc.)
- [ ] Distinguish **overfitting / underfitting** from symptoms

---

### 📚 Homework (see `Week2_HW1_Theory.ipynb`)

1. Calculate metrics **by hand** from a given confusion matrix
2. Essay: "Why high Recall is critical in medical diagnosis" — in IB Discuss [6] format
3. Questions on feature selection and dimensionality reduction
4. IB exam practice questions

---

> 🎓 **Final secret of Lesson 3:**
> In IB ML questions, **80% of marks** often come from **correct terminology** and **correct answer structure** for the command term.
> Learn the metric formulas **exactly**. In the exam, you may be given a confusion matrix and asked to calculate them. Without formulas, you lose the marks.
